In [1]:
import numpy as np

def euclidean_distance(p1, p2):
    """
    使用 NumPy 计算两个点之间的欧几里得距离。
    """
    p1 = np.asarray(p1)
    p2 = np.asarray(p2)
    return np.linalg.norm(p1 - p2)

In [2]:
rb_unity =  (-746.689575, 0,  -872.546753)
bridge_unity =  (-870.1338, 0, -1109.549)
euclidean_distance(rb_unity, bridge_unity)

267.22376722308894

In [3]:
rb = (-142.5, -160.022)
bridge = (-24.82, 79.91)
euclidean_distance(rb, bridge)

267.2376227704475

In [4]:
import numpy as np
def z2y(pos):
    x, y, z = pos
    return (x, z)

def compute_2d_tf_from_two_points(p1_A, p2_A, p1_B, p2_B):
    """
    计算同一水平面内两个坐标系之间的 2D 刚体变换（SE(2)），
    返回 4x4 齐次变换矩阵（适用于 ROS TF）。
    
    参数（支持 2D 或 3D 输入，但只使用 x, y）:
        p1_A, p2_A: 坐标系 A 中的起点和终点，shape (2,) or (3,)
        p1_B, p2_B: 坐标系 B 中的起点和终点，shape (2,) or (3,)
    
    返回:
        T_BA: 4x4 齐次变换矩阵，将 A 中的点变换到 B 坐标系
              [ R  t ]
              [ 0  1 ]
    """
    # 提取 x, y（兼容 2D/3D 输入）
    p1_A = np.asarray(p1_A)[:2]
    p2_A = np.asarray(p2_A)[:2]
    p1_B = np.asarray(p1_B)[:2]
    p2_B = np.asarray(p2_B)[:2]

    # 计算方向向量
    v_A = p2_A - p1_A
    v_B = p2_B - p1_B

    # 计算角度
    theta_A = np.arctan2(v_A[1], v_A[0])
    theta_B = np.arctan2(v_B[1], v_B[0])

    # 旋转角（从 A 到 B）
    theta = theta_B - theta_A

    # 构建 2D 旋转矩阵
    cos_t = np.cos(theta)
    sin_t = np.sin(theta)
    R_2d = np.array([[cos_t, -sin_t],
                     [sin_t,  cos_t]])

    # 计算平移（使 p1_A 映射到 p1_B）
    t_2d = p1_B - R_2d @ p1_A

    # 扩展为 3D 齐次矩阵（Z 不变，无绕 X/Y 旋转）
    T = np.eye(4)
    T[:2, :2] = R_2d
    T[:2, 3] = t_2d
    # T[2, 2] = 1 （默认）
    # T[2, 3] = 0 （假设同一水平面，z=0）

    return T

In [5]:
rb_unity = z2y(rb_unity)
bridge_unity = z2y(bridge_unity)
T_unity_to_real = compute_2d_tf_from_two_points(rb_unity, bridge_unity, rb, bridge)
T_unity_to_real

array([[-9.99707266e-01,  2.41946655e-02,  0.00000000e+00,
        -8.67860017e+02],
       [-2.41946655e-02, -9.99707266e-01,  0.00000000e+00,
        -1.05037923e+03],
       [ 0.00000000e+00,  0.00000000e+00,  1.00000000e+00,
         0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         1.00000000e+00]])

In [6]:
rb_unity_t = T_unity_to_real @ np.array([rb_unity[0], rb_unity[1], 0, 1])
bridge_unity_t = T_unity_to_real @ np.array([bridge_unity[0], bridge_unity[1], 0, 1])
rb_unity_t, bridge_unity_t

(array([-142.5  , -160.022,    0.   ,    1.   ]),
 array([-24.82610139,  79.89756018,   0.        ,   1.        ]))

In [7]:
# 读取json中的unity点位置信息，进行转换
import json
with open('waypoints.json', 'r') as f:
    waypoints_data = json.load(f)
waypoints_unity = waypoints_data['path']['waypoints']

# 读取tf json
with open('tf_unity_to_real.json', 'r') as f:
    tf_data = json.load(f)
T_unity_to_real = np.array(tf_data['T_unity_to_real'])

waypoints_real = []
for wp in waypoints_unity:
    wpp = wp['position']
    pos = z2y((wpp['x'], wpp['y'], wpp['z']))
    pos_h = np.array([pos[0], pos[1], 0, 1])
    pos_real_h = T_unity_to_real @ pos_h
    waypoints_real.append(pos_real_h)
waypoints_real

[array([ -83.94204965, -103.23193861,    0.        ,    1.        ]),
 array([-50.44656893, -60.93234414,   0.        ,   1.        ]),
 array([-6.63179017,  7.14246758,  0.        ,  1.        ]),
 array([ 3.69576103, 82.20331207,  0.        ,  1.        ]),
 array([-15.86108171,  76.22339027,   0.        ,   1.        ]),
 array([-24.82610139,  79.89756018,   0.        ,   1.        ])]

In [8]:
# 将tf矩阵保存为json

tf_data = {'T_unity_to_real': T_unity_to_real.tolist()}
with open('tf_unity_to_real.json', 'w') as f:
    json.dump(tf_data, f, indent=4)